question 
- 什么是4 bit量化
- lora到底在干什么
- unsloth库在干什么？

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
login(hf_token)

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = hf_token,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


d:\Anaconda\envs\nlp\lib\site-packages\unsloth_zoo\gradient_checkpointing.py:339: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  GPU_BUFFERS = tuple([torch.empty(2*256*2048, dtype = dtype, device = f"{DEVICE_TYPE}:{i}") for i in range(n_gpus)])


==((====))==  Unsloth 2025.5.9: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 5070. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [3]:
prompt_style = """Below is an instruction that describes a task, paired with 
an input that provides further context.  
Write a response that appropriately completes the request.  
Before answering, think carefully about the question and create a step-by-
step chain of thoughts to ensure a logical and accurate response. 
### Instruction: 
You are a medical expert with advanced knowledge in clinical reasoning, 
diagnostics, and treatment planning.  
Please answer the following medical question.  
### Question: 
{} 
### Response: 
<think>{}"""

question = "A 61-year-old woman with a long history of involuntary urine loss \
during activities like coughing or sneezing but no leakage at night undergoes \
a gynecological exam and Q-tip test. Based on these findings, what would \
cystometry most likely reveal about her residual volume and detrusor \
contractions?"

FastLanguageModel.for_inference(model)  
inputs = tokenizer([prompt_style.format(question, "")], 
return_tensors="pt").to("cuda") 

outputs = model.generate( 
    input_ids=inputs.input_ids, 
    attention_mask=inputs.attention_mask, 
    max_new_tokens=1200, 
    use_cache=True, 
) 
response = tokenizer.batch_decode(outputs) 
print(response[0].split("### Response:")[1])

 
<think>
Okay, so I'm trying to figure out what the cystometry would show for this 61-year-old woman. Let me start by breaking down the information given.

First, she has a history of involuntary urine loss when she coughs or sneezes. That makes me think of stress urinary incontinence, especially since she doesn't leak at night. So, maybe she's experiencing urge incontinence or some kind of overactive bladder.

She undergoes a gynecological exam and Q-tip test. From what I remember, the Q-tip test is used to check for urethral obstruction, which can cause symptoms like difficulty starting urination and a weak stream. If the test is positive, it suggests the urethra is blocked, but if it's negative, maybe the issue is elsewhere.

Now, the question is about what cystometry would reveal regarding residual volume and detrusor contractions. Cystometry is a test where they fill the bladder and measure how much it holds and how it contracts during filling. 

Residual volume is the amount of 

In [4]:
train_prompt_style = """Below is an instruction that describes a task, paired
with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-bystep chain of thoughts to ensure a logical and accurate response.
### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning,
diagnostics, and treatment planning.
Please answer the following medical question.
### Question:
{}
### Response:
<think>
{}
</think>
{}"""


EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT","en",
split = "train[0:500]",trust_remote_code=True)
print(dataset)


README.md:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

Dataset({
    features: ['Question', 'Complex_CoT', 'Response'],
    num_rows: 500
})


In [6]:
dataset = dataset.map(formatting_prompts_func, batched = True,)
print(dataset)
dataset["text"][0]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['Question', 'Complex_CoT', 'Response', 'text'],
    num_rows: 500
})


"Below is an instruction that describes a task, paired\nwith an input that provides further context.\nWrite a response that appropriately completes the request.\nBefore answering, think carefully about the question and create a step-by\x02step chain of thoughts to ensure a logical and accurate response.\n### Instruction:\nYou are a medical expert with advanced knowledge in clinical reasoning,\ndiagnostics, and treatment planning.\nPlease answer the following medical question.\n### Question:\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?\n### Response:\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swo

In [ ]:

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj","down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)